# Function 8: Black-box optimisation

This notebook documents 13 rounds of model fitting, candidate selection and evaluated outputs for Function 8. Alternative approaches are retained where they informed the weekly decisions.

**Main result:** The quadratic relationship recovered in Week 11 gave the maximum output of 10.0.

**Usage:** Run the setup and then the weekly sections in order. The initial datasets are in this folder; the full query history is in [`weekly_submissions.csv`](../submissions/weekly_submissions.csv). Some early searches generate large candidate grids and may require substantial memory.

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.optimize import differential_evolution
from scipy.spatial import cKDTree
from scipy.stats import norm, truncnorm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, RBF


def resolve_initial_path(filename: str) -> Path:
    """Find the initial dataset for Function 8."""
    for folder in (Path.cwd(), Path.cwd() / "function_8"):
        candidate = folder / filename
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"{filename} not found. Place it in the function_8/ folder."
    )

## Reusable routines

The helper functions below handle standard GP fitting, UCB scoring and recording evaluated queries. The model settings and experiments remain visible within each week. Global/local candidate generation is shared across the early rounds.

In [ ]:
def fit_gp_model(X, y, **settings):
    """Fit a GP using the explicit kernel and settings chosen for a given week."""
    gp = GaussianProcessRegressor(**settings)
    gp.fit(X, y)
    return gp


def ucb_score(mean, std, beta):
    """Upper Confidence Bound: predictive mean plus weighted uncertainty."""
    return mean + beta * std


def add_evaluation(X, y, x_observed, y_observed):
    """Add one completed black-box evaluation to the training history."""
    x_row = np.asarray(x_observed, dtype=float).reshape(1, -1)
    y_value = np.asarray(y_observed, dtype=float).reshape(1)
    if x_row.shape[1] != X.shape[1]:
        raise ValueError("The evaluated input has the wrong number of dimensions.")
    return np.vstack([X, x_row]), np.concatenate([y, y_value])

def sample_hybrid_candidates(best_x, n_global, n_local, seed, *,
                             local_scale=0.08, local_method="truncated"):
    """Combine uniform global samples with local samples near the current best.

    The local method is either "truncated" (truncated normal, inside [0, 1])
    or "clipped" (normal samples clipped to [0, 1]). Random number generation
    follows the same sequence for a given seed as the weekly experiments.
    """
    best_x = np.asarray(best_x, dtype=float).reshape(-1)
    n_features = best_x.size
    rng = np.random.default_rng(seed)

    X_global = rng.uniform(0, 1, size=(n_global, n_features))
    if local_method == "clipped":
        X_local = rng.normal(
            loc=best_x, scale=local_scale, size=(n_local, n_features)
        )
        X_local = np.clip(X_local, 0, 1)
    elif local_method == "truncated":
        X_local = np.empty((n_local, n_features))
        for j in range(n_features):
            a = (0 - best_x[j]) / local_scale
            b = (1 - best_x[j]) / local_scale
            X_local[:, j] = truncnorm.rvs(
                a, b, loc=best_x[j], scale=local_scale,
                size=n_local, random_state=rng
            )
    else:
        raise ValueError("local_method must be 'clipped' or 'truncated'.")

    return np.round(np.vstack([X_global, X_local]), 6)


## Starting data

In [2]:
X = np.load(resolve_initial_path("initial_inputs.npy"))
y = np.load(resolve_initial_path("initial_outputs.npy"))

## Week 1

In [8]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = 0.1

# UCB exploration parameter
beta = 1.96
# Configure the Gaussian Process
kernel = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds="fixed"
)

model = fit_gp_model(
    X, y,
    kernel=kernel,
    alpha=noise_assumption,
    normalize_y=True
)
# Generate candidate inputs
# Define the search range for each input variable

x1_values = np.linspace(0, 1, 8)
x2_values = np.linspace(0, 1, 8)
x3_values = np.linspace(0, 1, 8)
x4_values = np.linspace(0, 1, 8)
x5_values = np.linspace(0, 1, 8)
x6_values = np.linspace(0, 1, 8)
x7_values = np.linspace(0, 1, 8)
x8_values = np.linspace(0, 1, 8)

# Generate every combination of x1 and x2
X1_grid, X2_grid, X3_grid, X4_grid, X5_grid, X6_grid, X7_grid, X8_grid   = np.meshgrid(
    x1_values,
    x2_values,
    x3_values,
    x4_values,
    x5_values,
    x6_values,
    x7_values,
    x8_values
)

X_grid = np.column_stack([
    X1_grid.ravel(),
    X2_grid.ravel(),
    X3_grid.ravel(),
    X4_grid.ravel(),
    X5_grid.ravel(),
    X6_grid.ravel(),
    X7_grid.ravel(),
    X8_grid.ravel()
])
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_grid,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)
# Select the next query
best_index = np.argmax(UCB)

next_X = X_grid[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.         0.         0.         0.         0.42857143 0.85714286
 0.42857143 0.85714286]

Predicted mean:
8.53379558831494

Predicted standard deviation:
0.866632099749112

UCB:
10.2323945038232


### Evaluated query

**Input:** `(0.000000, 0.000000, 0.000000, 0.000000, 0.428571, 0.857143, 0.428571, 0.857143)`  
**Output:** `9.5766674702036`

In [ ]:
X_week1 = np.array([0., 0., 0., 0., 0.428571, 0.857143, 0.428571, 0.857143])
y_week1 = np.array([9.5766674702036])
X_new, y_new = add_evaluation(X, y, X_week1, y_week1)

In [12]:
X = X_new
y = y_new

### Increase β to favour exploration

In [13]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = 0.1

beta = 3

### Allow the kernel length scale to vary

In [14]:
# Configure the Gaussian Process
kernel = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.05, 1.5)
)

model = fit_gp_model(
    X, y,
    kernel=kernel,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=0.1)

Optimised kernel:
RBF(length_scale=0.915)


### Use random candidates instead of a grid

In [15]:
# Generate candidate inputs
n_candidates = 1000000
n_features = 8

np.random.seed(42)

X_candidates = np.random.uniform(
    low=0,
    high=1,
    size=(n_candidates, n_features)
)

X_candidates = np.round(X_candidates, 6)

In [16]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [17]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.213198 0.013852 0.044849 0.045754 0.827957 0.026294 0.068044 0.219353]

Predicted mean:
9.844440111369131

Predicted standard deviation:
0.591053845649789

UCB:
11.617601648318498


### Evaluated query

**Input:** `(0.213198, 0.013852, 0.044849, 0.045754, 0.827957, 0.026294, 0.068044, 0.219353)`  
**Output:** `9.6491146836756`

In [ ]:
X_week2 = np.array([0.213198, 0.013852, 0.044849, 0.045754, 0.827957, 0.026294, 0.068044, 0.219353])
y_week2 = np.array([9.6491146836756])
X_new, y_new = add_evaluation(X, y, X_week2, y_week2)

## Week 3

In [21]:
X = X_new
y = y_new

### Option 1: Isotropic GP with global/local sampling

In [22]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = 0.1

beta = 2.5

In [23]:
# Configure the Gaussian Process
kernel_iso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.05, 2)
)

model = fit_gp_model(
    X, y,
    kernel=kernel_iso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_iso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=0.1)

Optimised kernel:
RBF(length_scale=0.93)


In [24]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 8
n_global = 850000
n_local = 150000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=45,
    local_scale=0.08,
    local_method="truncated",
)

In [25]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [26]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.001228 0.513603 0.007073 0.139049 0.982577 0.490536 0.057175 0.897359]

Predicted mean:
9.753433744556357

Predicted standard deviation:
0.5286987791195056

UCB:
11.075180692355122


### Option 2: Anisotropic GP with global/local sampling · selected

In [27]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1]

beta = 2.5

In [28]:
# Configure the Gaussian Process
kernel_aniso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.05, 5)
)

model = fit_gp_model(
    X, y,
    kernel=kernel_aniso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_aniso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1])

Optimised kernel:
RBF(length_scale=[0.811, 1.11, 0.516, 2.11, 3.15, 5, 0.755, 5])


/opt/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 5 of parameter length_scale is close to the specified upper bound 5.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter length_scale is close to the specified upper bound 5.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [29]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 8
n_global = 850000
n_local = 150000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=45,
    local_scale=0.08,
    local_method="truncated",
)

In [30]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [31]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.046029 0.38137  0.027088 0.975285 0.949023 0.979835 0.252886 0.276403]

Predicted mean:
9.595336432264777

Predicted standard deviation:
0.3427912777865401

UCB:
10.452314626731127


### Evaluated query

**Input:** `(0.046029, 0.381370, 0.027088, 0.975285, 0.949023, 0.979835, 0.252886, 0.276403)`  
**Output:** `8.9703633006386`

In [ ]:
X_week3 = np.array([0.046029, 0.38137 , 0.027088, 0.975285, 0.949023, 0.979835, 0.252886, 0.276403])
y_week3 = np.array([8.9703633006386])
X_new, y_new = add_evaluation(X, y, X_week3, y_week3)

## Week 4

In [35]:
X = X_new
y = y_new

### Option 1: Keep isotropic GP, lower β to 1.5 · alternative

In [36]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = 0.1

beta = 1.5

In [37]:
# Configure the Gaussian Process
kernel_iso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.05, 2)
)

model = fit_gp_model(
    X, y,
    kernel=kernel_iso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_iso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=0.1)

Optimised kernel:
RBF(length_scale=0.931)


In [38]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 8
n_global = 850000
n_local = 150000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=46,
    local_scale=0.08,
    local_method="truncated",
)

In [39]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [40]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.087015 0.232238 0.13377  0.027237 0.704051 0.524022 0.032227 0.773708]

Predicted mean:
10.07258123551686

Predicted standard deviation:
0.3489155846701274

UCB:
10.595954612522052


### Option 2: Anisotropic GP with β = 1.5 and length-scale upper bound 10

In [41]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1]

bounds = np.array([(0.05, 5.0)] * 8)

bounds[5] = (0.05, 10.0)  # x6
bounds[7] = (0.05, 10.0)  # x8

beta = 1.5

In [42]:
# Configure the Gaussian Process
kernel_aniso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=bounds
)

model = fit_gp_model(
    X, y,
    kernel=kernel_aniso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_aniso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1])

Optimised kernel:
RBF(length_scale=[0.829, 1.11, 0.486, 1.51, 3.33, 10, 0.776, 10])


/opt/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 5 of parameter length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [43]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 8
n_global = 850000
n_local = 150000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=46,
    local_scale=0.08,
    local_method="truncated",
)

In [44]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [45]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.35498  0.062962 0.315531 0.003826 0.92998  0.037405 0.14811  0.160886]

Predicted mean:
9.932294324949037

Predicted standard deviation:
0.18238278515583675

UCB:
10.205868502682792


### Evaluated query

**Input:** `(0.354980, 0.062962, 0.315531, 0.003826, 0.929980, 0.037405, 0.148110, 0.160886)`  
**Output:** `9.490653902672399`

In [ ]:
X_week4 = np.array([0.35498 , 0.062962, 0.315531, 0.003826, 0.92998 , 0.037405, 0.14811, 0.160886])
y_week4 = np.array([9.490653902672399])
X_new, y_new = add_evaluation(X, y, X_week4, y_week4)

## Week 5

In [49]:
X = X_new
y = y_new

### Option 1: Anisotropic GP with β = 1 and length-scale upper bound 20

In [50]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1]

bounds = np.array([(0.05, 5.0)] * 8)

bounds[5] = (0.05, 20.0)  # x6
bounds[7] = (0.05, 20.0)  # x8

beta = 1

In [51]:
# Configure the Gaussian Process
kernel_aniso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=bounds
)

model = fit_gp_model(
    X, y,
    kernel=kernel_aniso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_aniso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1])

Optimised kernel:
RBF(length_scale=[0.785, 1.24, 0.544, 1.04, 3.61, 20, 0.834, 20])


/opt/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 5 of parameter length_scale is close to the specified upper bound 20.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter length_scale is close to the specified upper bound 20.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [52]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 8
n_global = 850000
n_local = 150000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=47,
    local_scale=0.08,
    local_method="truncated",
)

In [53]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [54]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.185701 0.219075 0.031497 0.306488 0.970828 0.629802 0.273031 0.238905]

Predicted mean:
9.920620786029506

Predicted standard deviation:
0.11747108011499496

UCB:
10.038091866144502


### Evaluated query

**Input:** `(0.185701, 0.219075, 0.031497, 0.306488, 0.970828, 0.629802, 0.273031, 0.238905)`  
**Output:** `9.8717966285815`

In [ ]:
X_week5 = np.array([0.185701, 0.219075, 0.031497, 0.306488, 0.970828, 0.629802, 0.273031, 0.238905])
y_week5 = np.array([9.8717966285815])
X_new, y_new = add_evaluation(X, y, X_week5, y_week5)

## Week 6

In [58]:
X = X_new
y = y_new

### Option 1: Anisotropic GP with β = 0.5 and length-scale upper bound 20

In [59]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1]

bounds = np.array([(0.05, 5.0)] * 8)

bounds[5] = (0.05, 20.0)  # x6
bounds[7] = (0.05, 20.0)  # x8

beta = 0.5

In [60]:
# Configure the Gaussian Process
kernel_aniso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=bounds
)

model = fit_gp_model(
    X, y,
    kernel=kernel_aniso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_aniso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1])

Optimised kernel:
RBF(length_scale=[0.801, 1.25, 0.541, 1.1, 3.72, 20, 0.842, 20])


/opt/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 5 of parameter length_scale is close to the specified upper bound 20.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter length_scale is close to the specified upper bound 20.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [61]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 8
n_global = 850000
n_local = 150000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=48,
    local_scale=0.08,
    local_method="truncated",
)

In [62]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [63]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.214912 0.197861 0.130028 0.253567 0.998174 0.608357 0.331247 0.289391]

Predicted mean:
9.903699293669245

Predicted standard deviation:
0.0439451927529796

UCB:
9.925671890045734


### Evaluated query

**Input:** `(0.214912, 0.197861, 0.130028, 0.253567, 0.998174, 0.608357, 0.331247, 0.289391)`  
**Output:** `9.8850966116569`

In [ ]:
X_week6 = np.array([0.214912, 0.197861, 0.130028, 0.253567, 0.998174, 0.608357, 0.331247, 0.289391])
y_week6 = np.array([9.8850966116569])
X_new, y_new = add_evaluation(X, y, X_week6, y_week6)

## Week 7

In [67]:
# Include Week 6 results
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).ravel()

assert X.shape == (46, 8)
assert y.shape == (46,)

# Length-scale bounds
bounds_rbf = np.array([(0.02, 5.0)] * 8)
bounds_rbf[[5, 7]] = (0.02, 20.0)

bounds_matern = np.array([(0.02, 5.0)] * 8)
bounds_matern[[4, 5, 7]] = (0.02, 20.0)

# Three surrogate models
kernels = [
    # Anisotropic RBF
    RBF(np.full(8, 0.8), bounds_rbf),

    # Anisotropic Matérn 5/2 with fitted amplitude
    ConstantKernel(1.0, (0.03, 100)) * Matern(
        np.full(8, 0.8),
        bounds_matern,
        nu=2.5
    ),

    # Isotropic RBF
    RBF(0.8, (0.02, 5.0))
]

weights = np.array([0.2, 0.5, 0.3])

models = [
    GaussianProcessRegressor(
        kernel=k,
        alpha=1e-10,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    ).fit(X, y)
    for k in kernels
]

for model in models:
    print(model.kernel_)

# Combine predictions and model disagreement
def ensemble_predict(points):
    predictions = [
        model.predict(points, return_std=True)
        for model in models
    ]

    means = np.array([p[0] for p in predictions])
    variances = np.array([p[1]**2 for p in predictions])

    mu = np.average(
        means, axis=0, weights=weights
    )

    variance = np.average(
        variances + (means - mu)**2,
        axis=0,
        weights=weights
    )

    return mu, np.sqrt(variance)

# Local trust region around the best observed input
best_x = X[np.argmax(y)]
radius = 0.25

lower = np.maximum(0, best_x - radius)
upper = np.minimum(1, best_x + radius)

# Retain the promising high-x5 region
lower[4] = max(lower[4], 0.95)

search_bounds = list(zip(lower, upper))

# Acquisition parameters
beta = 0.5
tree = cKDTree(X)
min_distance = 0.05

def negative_ucb(points):
    candidates = points.T

    mu, std = ensemble_predict(candidates)
    distances = tree.query(candidates)[0]

    penalty = 50 * np.maximum(
        0, min_distance - distances
    )

    return -(ucb_score(mu, std, beta)) + penalty

# Optimise the acquisition function
result = differential_evolution(
    negative_ucb,
    bounds=search_bounds,
    seed=48,
    popsize=15,
    maxiter=130,
    tol=1e-7,
    vectorized=True,
    updating="deferred",
    polish=False
)

next_X = result.x
mu, std = ensemble_predict(next_X.reshape(1, -1))

print("Week 7 query:", np.round(next_X, 6))
print("Predicted mean:", mu[0])
print("Model-averaged std:", std[0])
print("UCB:", mu[0] + beta * std[0])
print("Nearest distance:", tree.query(next_X)[0])

/opt/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 5 of parameter length_scale is close to the specified upper bound 20.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter length_scale is close to the specified upper bound 20.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


RBF(length_scale=[0.803, 1.27, 0.562, 1.11, 3.83, 20, 0.859, 20])
3.89**2 * Matern(length_scale=[3.38, 5, 2.84, 4.47, 17.6, 5.2, 3.52, 20], nu=2.5)
RBF(length_scale=0.987)
Week 7 query: [0.099658 0.283905 0.092691 0.179222 0.950003 0.465735 0.206416 0.539391]
Predicted mean: 9.988498310709035
Model-averaged std: 0.16039297806789046
UCB: 10.06869479974298
Nearest distance: 0.3582629403647214


/opt/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k2__length_scale is close to the specified upper bound 5.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter k2__length_scale is close to the specified upper bound 20.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


### Evaluated query

**Input:** `(0.099658, 0.283905, 0.092691, 0.179222, 0.950003, 0.465735, 0.206416, 0.539391)`  
**Output:** `9.9641651918904`

In [ ]:
X_week7 = np.array([0.099658, 0.283905, 0.092691, 0.179222, 0.950003, 0.465735, 0.206416, 0.539391])
y_week7 = np.array([9.9641651918904])
X_new, y_new = add_evaluation(X, y, X_week7, y_week7)

## Week 8

In [71]:
# All observations through Week 7
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).ravel()

assert X.shape == (47, 8)
assert y.shape == (47,)

# Anisotropic RBF bounds
bounds_rbf = np.array([(0.02, 5.0)] * 8)
bounds_rbf[[5, 7]] = (0.02, 20.0)

# Expanded Matérn length-scale bounds
bounds_matern = np.array([(0.02, 20.0)] * 8)

kernels = [
    # 20%: anisotropic RBF
    RBF(
        length_scale=np.full(8, 0.8),
        length_scale_bounds=bounds_rbf
    ),

    # 50%: anisotropic Matérn 5/2
    # with fitted signal amplitude
    ConstantKernel(
        1.0,
        (0.03, 100.0)
    ) * Matern(
        length_scale=np.full(8, 0.8),
        length_scale_bounds=bounds_matern,
        nu=2.5
    ),

    # 30%: isotropic RBF
    RBF(
        length_scale=0.8,
        length_scale_bounds=(0.02, 5.0)
    )
]

weights = np.array([0.2, 0.5, 0.3])

models = []

for kernel in kernels:
    gp = GaussianProcessRegressor(
        kernel=kernel,
        alpha=1e-10,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    gp.fit(X, y)
    models.append(gp)
    print("Fitted:", gp.kernel_)

# Ensemble mean and uncertainty
def ensemble_predict(points):
    predictions = [
        gp.predict(points, return_std=True)
        for gp in models
    ]

    means = np.array([p[0] for p in predictions])
    variances = np.array([p[1]**2 for p in predictions])

    mean = np.average(
        means,
        axis=0,
        weights=weights
    )

    variance = np.average(
        variances + (means - mean)**2,
        axis=0,
        weights=weights
    )

    return mean, np.sqrt(
        np.maximum(variance, 0)
    )

# Trust region around current optimum
best_x = X[np.argmax(y)]
radius = 0.35

lower = np.maximum(0, best_x - radius)
upper = np.minimum(1, best_x + radius)

# Retain the promising high-x5 region
lower[4] = max(lower[4], 0.95)

bounds = list(zip(lower, upper))

# Acquisition function
beta = 0.5
min_distance = 0.05
tree = cKDTree(X)

def negative_ucb(points):
    candidates = points.T

    mu, std = ensemble_predict(candidates)
    distances = tree.query(candidates)[0]

    penalty = 1000 * np.maximum(
        0, min_distance - distances
    )

    return -(ucb_score(mu, std, beta)) + penalty

# Multiple independent acquisition searches
solutions = []

for seed in [42, 48, 59]:
    result = differential_evolution(
        negative_ucb,
        bounds=bounds,
        seed=seed,
        popsize=13,
        maxiter=160,
        tol=2e-7,
        vectorized=True,
        updating="deferred",
        polish=False
    )

    solutions.append(result)

best_result = min(
    solutions,
    key=lambda r: r.fun
)

next_X = np.round(best_result.x, 6)

mu, std = ensemble_predict(
    next_X.reshape(1, -1)
)

print("Week 8 query:", next_X)
print("Predicted mean:", mu[0])
print("Model-averaged std:", std[0])
print("UCB:", mu[0] + beta * std[0])
print(
    "Nearest distance:",
    tree.query(next_X)[0]
)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 5 of parameter length_scale is close to the specified upper bound 20.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter length_scale is close to the specified upper bound 20.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Fitted: RBF(length_scale=[0.827, 1.2, 0.559, 1.11, 4.04, 20, 0.865, 20])


/opt/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 4 of parameter k2__length_scale is close to the specified upper bound 20.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter k2__length_scale is close to the specified upper bound 20.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Fitted: 5.49**2 * Matern(length_scale=[4.21, 7.05, 3.64, 5.46, 20, 6.45, 4.38, 20], nu=2.5)
Fitted: RBF(length_scale=1)
Week 8 query: [0.101255 0.250734 0.083321 0.205013 0.999999 0.494446 0.238621 0.889389]
Predicted mean: 9.938994483506097
Model-averaged std: 0.16076412020189065
UCB: 10.019376543607041
Nearest distance: 0.35876945493868345


### Evaluated query

**Input:** `(0.101255, 0.250734, 0.083321, 0.205013, 0.999999, 0.494446, 0.238621, 0.889389)`  
**Output:** `9.9488978843714`

In [ ]:
X_week8 = np.array([0.101255, 0.250734, 0.083321, 0.205013, 0.999999, 0.494446, 0.238621, 0.889389])
y_week8 = np.array([9.9488978843714])
X_new, y_new = add_evaluation(X, y, X_week8, y_week8)

## Week 9

In [75]:
# All observations through Week 8
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).ravel()

assert X.shape == (48, 8)
assert y.shape == (48,)
# Define GP kernels
# Anisotropic RBF length-scale bounds
rbf_bounds = np.array(
    [(0.02, 5.0)] * 8
)
rbf_bounds[[5, 7]] = (0.02, 20.0)

# Anisotropic Matérn bounds
matern_bounds = np.array(
    [(0.02, 20.0)] * 8
)

kernels = [
    # Model 1: anisotropic RBF
    RBF(
        length_scale=np.full(8, 0.8),
        length_scale_bounds=rbf_bounds
    ),

    # Model 2: anisotropic Matérn 5/2
    # with fitted signal amplitude
    ConstantKernel(
        1.0,
        constant_value_bounds=(0.03, 100.0)
    ) * Matern(
        length_scale=np.full(8, 0.8),
        length_scale_bounds=matern_bounds,
        nu=2.5
    ),

    # Model 3: isotropic RBF
    RBF(
        length_scale=0.8,
        length_scale_bounds=(0.02, 5.0)
    )
]

weights = np.array([0.2, 0.5, 0.3])

models = []

for kernel in kernels:
    gp = GaussianProcessRegressor(
        kernel=kernel,
        alpha=1e-10,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    gp.fit(X, y)
    models.append(gp)

    print("Fitted kernel:", gp.kernel_)
# Ensemble predictions
def ensemble_predict(points):

    predictions = [
        gp.predict(points, return_std=True)
        for gp in models
    ]

    means = np.array([
        prediction[0]
        for prediction in predictions
    ])

    variances = np.array([
        prediction[1]**2
        for prediction in predictions
    ])

    mean = np.average(
        means,
        axis=0,
        weights=weights
    )

    variance = np.average(
        variances + (means - mean)**2,
        axis=0,
        weights=weights
    )

    return mean, np.sqrt(
        np.maximum(variance, 0)
    )
# Trust region
best_x = X[np.argmax(y)]

radius = 0.35

lower = np.maximum(
    0,
    best_x - radius
)

upper = np.minimum(
    1,
    best_x + radius
)

# Continue searching the strong x5 region
lower[4] = max(lower[4], 0.95)

bounds = list(zip(lower, upper))
# Acquisition function
beta = 0.5
min_distance = 0.05

tree = cKDTree(X)

def negative_ucb(points):

    points = np.asarray(points)

    candidates = (
        points.T
        if points.ndim == 2
        else points.reshape(1, -1)
    )

    mu, std = ensemble_predict(
        candidates
    )

    distances = tree.query(
        candidates
    )[0]

    ucb = ucb_score(mu, std, beta)

    penalty = 2000 * np.maximum(
        0,
        min_distance - distances
    )

    scores = -ucb + penalty

    return (
        scores
        if points.ndim == 2
        else scores[0]
    )
# Optimise ensemble GP-UCB
solutions = []

for seed in [48, 59, 101]:

    result = differential_evolution(
        negative_ucb,
        bounds=bounds,
        seed=seed,
        popsize=13,
        maxiter=170,
        tol=1e-7,
        vectorized=True,
        updating="deferred",
        polish=False
    )

    if tree.query(result.x)[0] >= (
        min_distance - 1e-4
    ):
        solutions.append(result)

if not solutions:
    raise RuntimeError(
        "No feasible candidate was found."
    )

best_result = min(
    solutions,
    key=lambda result: result.fun
)

next_X = np.round(
    best_result.x,
    6
)
# Report Week 9 recommendation
mu, std = ensemble_predict(
    next_X.reshape(1, -1)
)

print("Week 9 query:", next_X)
print("Predicted mean:", mu[0])
print("Ensemble std:", std[0])
print("GP-UCB:", mu[0] + beta * std[0])
print(
    "Nearest observation distance:",
    tree.query(next_X)[0]
)
print("Current best:", y.max())

/opt/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 5 of parameter length_scale is close to the specified upper bound 20.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter length_scale is close to the specified upper bound 20.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Fitted kernel: RBF(length_scale=[0.835, 1.18, 0.566, 1.15, 3.94, 20, 0.835, 20])


/opt/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter k2__length_scale is close to the specified upper bound 20.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Fitted kernel: 6.27**2 * Matern(length_scale=[4.98, 7.09, 4.09, 6.84, 11.8, 6.86, 5.09, 20], nu=2.5)
Fitted kernel: RBF(length_scale=1.02)
Week 9 query: [0.109846 0.176828 0.140311 0.15911  0.95     0.552983 0.226018 0.702463]
Predicted mean: 9.965622317069597
Ensemble std: 0.060864956761560035
GP-UCB: 9.996054795450377
Nearest observation distance: 0.22097593832361023
Current best: 9.9641651918904


### Evaluated query

**Input:** `(0.109846, 0.176828, 0.140311, 0.159110, 0.950000, 0.552983, 0.226018, 0.702463)`  
**Output:** `9.9822234911471`

In [ ]:
X_week9 = np.array([0.109846, 0.176828, 0.140311, 0.15911 , 0.95 , 0.552983, 0.226018, 0.702463])
y_week9 = np.array([9.9822234911471])
X_new, y_new = add_evaluation(X, y, X_week9, y_week9)

## Week 10

In [79]:
# Observations through Week 9
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).ravel()

assert X.shape == (49, 8)
assert y.shape == (49,)
# Define GP models
rbf_bounds = np.array(
    [(0.02, 5.0)] * 8
)
rbf_bounds[[5, 7]] = (0.02, 20.0)

matern_bounds = np.array(
    [(0.02, 20.0)] * 8
)

kernels = [
    RBF(
        length_scale=np.full(8, 0.8),
        length_scale_bounds=rbf_bounds
    ),

    ConstantKernel(
        1.0,
        constant_value_bounds=(0.03, 100.0)
    ) * Matern(
        length_scale=np.full(8, 0.8),
        length_scale_bounds=matern_bounds,
        nu=2.5
    ),

    RBF(
        length_scale=0.8,
        length_scale_bounds=(0.02, 5.0)
    )
]

models = []

for kernel in kernels:

    gp = GaussianProcessRegressor(
        kernel=kernel,
        alpha=1e-10,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    gp.fit(X, y)
    models.append(gp)

    print("Fitted kernel:", gp.kernel_)
# Weighted ensemble predictions
weights = np.array([0.2, 0.5, 0.3])

def ensemble_predict(points):

    predictions = [
        gp.predict(points, return_std=True)
        for gp in models
    ]

    means = np.array([
        result[0] for result in predictions
    ])

    variances = np.array([
        result[1]**2 for result in predictions
    ])

    mean = np.average(
        means,
        axis=0,
        weights=weights
    )

    variance = np.average(
        variances + (means - mean)**2,
        axis=0,
        weights=weights
    )

    return mean, np.sqrt(
        np.maximum(variance, 0)
    )
# Updated trust region
best_x = X[np.argmax(y)]
radius = 0.35

lower = np.maximum(
    0, best_x - radius
)

upper = np.minimum(
    1, best_x + radius
)

# No additional x5 restriction
bounds = list(zip(lower, upper))
# GP-UCB acquisition function
beta = 0.5
min_distance = 0.05

tree = cKDTree(X)

def negative_ucb(points):

    points = np.asarray(points)

    candidates = (
        points.T
        if points.ndim == 2
        else points.reshape(1, -1)
    )

    mu, std = ensemble_predict(
        candidates
    )

    distances = tree.query(candidates)[0]

    ucb = ucb_score(mu, std, beta)

    penalty = 500 * np.maximum(
        0,
        min_distance + 0.0005 - distances
    )

    scores = -ucb + penalty

    return (
        scores
        if points.ndim == 2
        else scores[0]
    )
# Optimise acquisition
solutions = []

for seed in [42, 59, 111]:

    result = differential_evolution(
        negative_ucb,
        bounds=bounds,
        seed=seed,
        popsize=12,
        maxiter=160,
        tol=1e-7,
        vectorized=True,
        updating="deferred",
        polish=True
    )

    if tree.query(result.x)[0] >= min_distance:
        solutions.append(result)

if not solutions:
    raise RuntimeError(
        "No feasible candidate was found."
    )

best_result = min(
    solutions,
    key=lambda result: result.fun
)

next_X = np.round(
    best_result.x, 6
)
# Report Week 10 recommendation
mu, std = ensemble_predict(
    next_X.reshape(1, -1)
)

print("Week 10 query:", next_X)
print("Predicted mean:", mu[0])
print("Ensemble std:", std[0])
print("GP-UCB:", mu[0] + beta * std[0])

print(
    "Nearest observation distance:",
    tree.query(next_X)[0]
)

print("Current best:", np.max(y))

/opt/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 4 of parameter length_scale is close to the specified upper bound 5.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter length_scale is close to the specified upper bound 20.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Fitted kernel: RBF(length_scale=[1.02, 1.23, 0.58, 2.49, 5, 1.13, 0.881, 20])


/opt/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter k2__length_scale is close to the specified upper bound 20.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Fitted kernel: 6.59**2 * Matern(length_scale=[5.23, 7.5, 4.28, 7.14, 11.8, 7.04, 5.39, 20], nu=2.5)
Fitted kernel: RBF(length_scale=1.03)
Week 10 query: [0.121618 0.212301 0.127569 0.105092 0.720801 0.503008 0.199897 0.694744]
Predicted mean: 9.98779476088041
Ensemble std: 0.06063033627422568
GP-UCB: 10.018109929017523
Nearest observation distance: 0.24545628618758158
Current best: 9.9822234911471


### Evaluated query

**Input:** `(0.121618, 0.212301, 0.127569, 0.105092, 0.720801, 0.503008, 0.199897, 0.694744)`  
**Output:** `9.9891064991679`

In [ ]:
X_week10 = np.array([0.121618, 0.212301, 0.127569, 0.105092, 0.720801, 0.503008, 0.199897, 0.694744])
y_week10 = np.array([9.9891064991679])
X_new, y_new = add_evaluation(X, y, X_week10, y_week10)

## Week 11

In [82]:
# All observations through Week 10
X_all = np.asarray(X_new, dtype=float)
y_all = np.asarray(y_new, dtype=float).ravel()

assert X_all.shape == (50, 8)
assert y_all.shape == (50,)
# Add linear and squared features
def quadratic_features(X):
    X = np.atleast_2d(X)

    return np.column_stack([
        np.ones(len(X)),
        X,
        X**2
    ])

Phi = quadratic_features(X_all)
# Out-of-sample validation
# Fit using only the 40 initial observations
coeff_initial = np.linalg.lstsq(
    Phi[:40],
    y_all[:40],
    rcond=None
)[0]

# Predict the ten weekly submissions
predicted_weekly = (
    Phi[40:] @ coeff_initial
)

validation_mae = np.mean(
    np.abs(predicted_weekly - y_all[40:])
)

print("Weekly holdout MAE:", validation_mae)

print("\nActual vs predicted:")
for actual, predicted in zip(
    y_all[40:],
    predicted_weekly
):
    print(f"{actual:.9f} | {predicted:.9f}")
# Fit using all 50 observations
coefficients = np.linalg.lstsq(
    Phi,
    y_all,
    rcond=None
)[0]

intercept = coefficients[0]
linear = coefficients[1:9]
quadratic = coefficients[9:17]

print("\nLinear coefficients:")
print(linear)

print("\nSquared coefficients:")
print(quadratic)
# Calculate quadratic maximum
assert np.all(quadratic < 0), (
    "Quadratic model is not concave "
    "in every input dimension."
)

x_opt = -linear / (2 * quadratic)
x_opt = np.clip(x_opt, 0, 1)
x_opt = np.round(x_opt, 6)

predicted_max = (
    quadratic_features(x_opt)
    @ coefficients
)[0]
# Report proposed input
print("\nWeek 11 input:")
print(" ".join(f"{v:.6f}" for v in x_opt))

print("\nPredicted output:", predicted_max)
print("Current best:", np.max(y_all))
print(
    "Predicted improvement:",
    predicted_max - np.max(y_all)
)

Weekly holdout MAE: 1.6697754290362355e-14

Actual vs predicted:
9.576667470 | 9.576667470
9.649114684 | 9.649114684
8.970363301 | 8.970363301
9.490653903 | 9.490653903
9.871796629 | 9.871796629
9.885096612 | 9.885096612
9.964165192 | 9.964165192
9.948897884 | 9.948897884
9.982223491 | 9.982223491
9.989106499 | 9.989106499

Linear coefficients:
[0.4  0.3  0.78 0.3  0.8  1.   0.8  0.12]

Squared coefficients:
[-2.  -1.  -3.  -1.  -0.5 -1.  -2.  -0.1]

Week 11 input:
0.100000 0.150000 0.130000 0.150000 0.800000 0.500000 0.200000 0.600000

Predicted output: 9.999999999999988
Current best: 9.9891064991679
Predicted improvement: 0.010893500832088066


### Evaluated query

**Input:** `(0.100000, 0.150000, 0.130000, 0.150000, 0.800000, 0.500000, 0.200000, 0.600000)`  
**Output:** `10.0`

In [ ]:
X_week11 = np.array([0.1 , 0.15, 0.13, 0.15, 0.8 , 0.5 , 0.2 , 0.6])
y_week11 = np.array([10.0])
X_new = np.vstack([
    X,
    X_week10.reshape(1, -1),
    X_week11.reshape(1, -1)
])
y_new = np.concatenate([
    y,
    y_week10.ravel(),
    y_week11.ravel()
])

## Week 12

In [85]:
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).ravel()

assert X.shape == (51, 8)

# Recovered function parameters
centre = np.array([
    0.10, 0.15, 0.13, 0.15,
    0.80, 0.50, 0.20, 0.60
])

weights = np.array([
    2.0, 1.0, 3.0, 1.0,
    0.5, 1.0, 2.0, 0.1
])

def recovered_function(X):
    X = np.atleast_2d(X)
    return 10.0 - np.sum(
        weights * (X - centre)**2,
        axis=1
    )

# Verify against all 51 observations
predictions = recovered_function(X)

errors = np.abs(y - predictions)

print("Maximum absolute error:", errors.max())
print("Mean absolute error:", errors.mean())
print("Recovered maximum:", recovered_function(centre)[0])

# Optional Week 12 validation experiment
next_X = centre.copy()
next_X[0] += 0.10
next_X[2] += 0.10

print("\nWeek 12 validation input:")
print(" ".join(f"{v:.6f}" for v in next_X))

print("Predicted output:", recovered_function(next_X)[0])

Maximum absolute error: 1.7763568394002505e-15
Mean absolute error: 1.5673736818237505e-16
Recovered maximum: 10.0

Week 12 validation input:
0.200000 0.150000 0.230000 0.150000 0.800000 0.500000 0.200000 0.600000
Predicted output: 9.95


### Evaluated query

**Input:** `(0.200000, 0.150000, 0.230000, 0.150000, 0.800000, 0.500000, 0.200000, 0.600000)`  
**Output:** `9.95`

In [ ]:
X_week12 = np.array([0.2 , 0.15, 0.23, 0.15, 0.8 , 0.5 , 0.2 , 0.6])
y_week12 = np.array([9.95])
X_new, y_new = add_evaluation(X, y, X_week12, y_week12)

## Week 13

In [88]:
# All observations through Week 12
X_all = np.asarray(X_new, dtype=float)
y_all = np.asarray(y_new, dtype=float).ravel()

assert X_all.shape == (52, 8)
assert y_all.shape == (52,)
# Recovered exact function
centre = np.array([
    0.10, 0.15, 0.13, 0.15,
    0.80, 0.50, 0.20, 0.60
])

weights = np.array([
    2.0, 1.0, 3.0, 1.0,
    0.5, 1.0, 2.0, 0.1
])

def recovered_function(X):
    X = np.atleast_2d(X)

    return 10 - np.sum(
        weights * (X - centre)**2,
        axis=1
    )
# Verify against all 52 observations
predictions = recovered_function(X_all)
errors = np.abs(predictions - y_all)

print("Observations:", len(y_all))
print("Maximum absolute error:", errors.max())
print("Mean absolute error:", errors.mean())

print("\nWeek 12 actual:", y_all[-1])
print("Week 12 predicted:", predictions[-1])
# Verify global maximum
print("\nExact model optimum:")
print(" ".join(f"{v:.6f}" for v in centre))
print("Maximum output:", recovered_function(centre)[0])
# Final submission if a new input is needed
next_X = centre.copy()
next_X[7] = 0.601

print("\nAlternative Week 13 input:")
print(" ".join(f"{v:.6f}" for v in next_X))

print(
    "Predicted output:",
    recovered_function(next_X)[0]
)

print(
    "Difference from global maximum:",
    10 - recovered_function(next_X)[0]
)

Observations: 52
Maximum absolute error: 1.7763568394002505e-15
Mean absolute error: 1.5372318802502167e-16

Week 12 actual: 9.95
Week 12 predicted: 9.95

Exact model optimum:
0.100000 0.150000 0.130000 0.150000 0.800000 0.500000 0.200000 0.600000
Maximum output: 10.0

Alternative Week 13 input:
0.100000 0.150000 0.130000 0.150000 0.800000 0.500000 0.200000 0.601000
Predicted output: 9.9999999
Difference from global maximum: 9.99999993922529e-08


### Week 13 evaluated result

**Input:** `(0.100000, 0.150000, 0.130000, 0.150000, 0.800000, 0.500000, 0.200000, 0.601000)`  
**Output:** `9.9999999`

The query-selection calculations above use results available through Week 12. The full record is in [`weekly_submissions.csv`](../submissions/weekly_submissions.csv).